In [ ]:
%pip install numpy matplotlib tensorflow scikit-learn pillow seaborn

In [ ]:
import os
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

zip_path = "cifake.zip"
extract_folder = "cifake_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)
    

print("Dataset extracted successfully")


In [ ]:
print(os.listdir("cifake_data"))

In [ ]:
print("Train folders:", os.listdir("cifake_data/train"))
print("Test folders:", os.listdir("cifake_data/test"))


In [ ]:
SEED = 42
IMAGE_SIZE = (32, 32)
BATCH_SIZE= 128
VALIDATION_RATIO = 0.20
EPOCHS = 20

TRAIN_PATH = "cifake_data/train"
TEST_PATH = "cifake_data/test"

LABEL_NAMES = ["REAL", "FAKE"]

np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH, class_names=LABEL_NAMES, labels="inferred", label_mode="binary",
    validation_split=VALIDATION_RATIO, subset="training", seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
validation_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH, class_names=LABEL_NAMES, labels="inferred", label_mode="binary",
    validation_split=VALIDATION_RATIO, subset="validation", seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=True,
)
test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH, class_names=LABEL_NAMES, labels="inferred", label_mode="binary",
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)


In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
train_data=train_data.prefetch(AUTOTUNE)
validation_data=validation_data.prefetch(AUTOTUNE)
test_data=test_data.prefetch(AUTOTUNE)


In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in train_data.take(1):
    for index in range(min(12, len(images))):
        plt.subplot(3, 4, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        label = int(labels[index].numpy().item())
        plt.title("AI-GENERATED" if label == 1 else "REAL")
        plt.axis("off")
plt.suptitle("Sample CIFAKE Images", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            fill_mode="reflect"
        ),
    ],
    name="data_augmentation"
)

In [ ]:
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1.0 / 255, name="normalization")(x)
x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="ai_probability")(x)
model = tf.keras.Model(inputs, outputs, name="cifake_detector")

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
         tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
         tf.keras.metrics.AUC(name="AUC"),
    ],
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
epochs_completed = range(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(epochs_completed,history.history["accuracy"],label="Training")
plt.plot(epochs_completed,history.history["val_accuracy"],label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch") 
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_completed, history.history["loss"], label="Training")
plt.plot(epochs_completed, history.history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\nEvaluating the model on unseen test images...")
test_results = model.evaluate(test_data, return_dict=True, verbose=1)
print("\nTEST RESULTS")
print("-" * 40)
for metric_name, metric_value in test_results.items():
    print(f"{metric_name.upper():10s}: {metric_value:.4f}")

In [ ]:
actual_labels = np.concatenate(
    [labels.numpy().reshape(-1) for _, labels in test_data]
).astype(int)
ai_scores = model.predict(test_data, verbose=1).reshape(-1)
THRESHOLD = 0.50
predicted_labels = (ai_scores >= THRESHOLD).astype(int)

In [ ]:
precision = precision_score(actual_labels, predicted_labels)
recall = recall_score(actual_labels, predicted_labels)
f1 = f1_score(actual_labels, predicted_labels)
roc_auc = roc_auc_score(actual_labels, ai_scores)

print("\nAI-GENERATED IMAGE METRICS")
print("-" * 40)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

In [ ]:
print("\nCLASSIFICATION REPORT")
print("-" * 40)
print(
    classification_report(
        actual_labels,
        predicted_labels,
        target_names=["REAL", "AI-GENERATED"],
        digits=4,
    )
)

matrix = confusion_matrix(actual_labels, predicted_labels)
plt.figure(figsize=(7, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=["Predicted REAL", "Predicted AI"],
    yticklabels=["Actual REAL", "Actual AI"],
)
plt.title("CIFAKE Confusion Matrix")
plt.xlabel("Model prediction")
plt.ylabel("True class")
plt.tight_layout()
plt.show()

print("\nCONFUSION MATRIX")
print("-" * 40)
print(f"Real images correctly classified:       {matrix[0, 0]:,}")
print(f"Real images incorrectly classified as AI:  {matrix[0, 1]:,}")
print(f"AI images classified as real:  {matrix[1, 0]:,}")
print(f"AI images detected correctly:           {matrix[1, 1]:,}")

In [ ]:
def predict_image(image_path, threshold=0.50):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Could not find: {image_path.resolve()}")
    original = tf.keras.utils.load_img(image_path)
    resized = tf.keras.utils.load_img(
        image_path,
        target_size=IMAGE_SIZE,
        color_mode="rgb",
    )
    image_array = tf.keras.utils.img_to_array(resized)
    image_array = np.expand_dims(image_array, axis=0)
    probability_ai = float(model.predict(image_array, verbose=0)[0][0])

    if probability_ai >= threshold:
        prediction = "AI-GENERATED"
        confidence = probability_ai
    else:
        prediction = "REAL"
        confidence = 1 - probability_ai
        
    print("\nCUSTOM IMAGE PREDICTION")
    print("-" * 40)
    print("Image:", image_path.name)
    print("Prediction:", prediction)
    print(f"Confidence: {confidence:.2%}")
    print(f"Probability of AI-generated: {probability_ai:.2%}")
    print(f"Probability of real: {1 - probability_ai:.2%}")

    plt.figure(figsize=(6, 6))
    plt.imshow(original)
    plt.title(f"{prediction}\nConfidence: {confidence:.2%}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return {
        "prediction": prediction,
        "confidence": confidence,
        "ai_probability": probability_ai,
        "real_probability": 1 - probability_ai,
    }


model.save("cifake_detector.keras")
print("\nModel saved as cifake_detector.keras")

CUSTOM_IMAGE_PATH = "test_image.jpeg"
if CUSTOM_IMAGE_PATH is not None:
    predict_image(CUSTOM_IMAGE_PATH)
else:
    print(
        "\nTo test your own image, set CUSTOM_IMAGE_PATH to its filename "
        "and rerun this cell."
    )